# LastFM (HetRec 2011) — Behavior Feature Computation

This notebook computes the **6 behavior features** for the LastFM dataset:

1. Activity
2. Popularity
3. Influence
4. Community Participation
5. Reciprocity
6. Reputation / Trustworthiness


In [ ]:
# If needed, install dependencies first (uncomment):
# !pip install pandas networkx python-louvain --quiet

import pandas as pd
import networkx as nx
import community as community_louvain  # from python-louvain package

DATA_DIR = "C:\Users\lavis\OneDrive\Documents\FinalYearProject\PHASE2"  # change this to the folder containing the .dat files


## Step 1 — Load the raw files

In [ ]:
user_friends = pd.read_csv(DATA_DIR + "user_friends.dat", sep="\t")
user_artists = pd.read_csv(DATA_DIR + "user_artists.dat", sep="\t")
user_tagged  = pd.read_csv(DATA_DIR + "user_taggedartists.dat", sep="\t")

print("user_friends:", user_friends.shape)
print("user_artists:", user_artists.shape)
print("user_tagged:", user_tagged.shape)

user_friends.head()


## Feature 1 — Activity

How engaged a user is on the platform: number of distinct artists listened to, plus number of tags they've applied.
Higher activity = more behavioral data available for that user, which matters for recommendation quality (cold-start risk if low).


In [ ]:
artists_listened = user_artists.groupby("userID")["artistID"].nunique().rename("artists_listened")
tags_applied     = user_tagged.groupby("userID")["tagID"].count().rename("tags_applied")

activity = pd.concat([artists_listened, tags_applied], axis=1).fillna(0)
activity["activity_score"] = activity["artists_listened"] + activity["tags_applied"]

activity.sort_values("activity_score", ascending=False).head(10)


## Step 2 — Build the friend network graph

Needed for Popularity, Influence, Community Participation, and Reciprocity.


In [ ]:
G = nx.from_pandas_edgelist(user_friends, source="userID", target="friendID", create_using=nx.Graph())
print("Nodes:", G.number_of_nodes(), "| Edges:", G.number_of_edges())


## Feature 2 — Popularity

Number of friends a user has (degree in the friend network). This is your "followers/listeners" equivalent for LastFM.


In [ ]:
popularity = pd.Series(dict(G.degree()), name="popularity_degree")
popularity.sort_values(ascending=False).head(10)


## Feature 3 — Influence

Two standard measures:
- **PageRank**: how influential a user is based on who is connected to them (not just how many friends, but how well-connected those friends are)
- **Betweenness Centrality**: how often a user sits "between" other users on the shortest path — i.e. how much of a bridge/connector they are


In [ ]:
pagerank    = pd.Series(nx.pagerank(G), name="pagerank")
betweenness = pd.Series(nx.betweenness_centrality(G), name="betweenness")

influence = pd.concat([pagerank, betweenness], axis=1)
influence.sort_values("pagerank", ascending=False).head(10)


## Feature 4 — Community Participation

Uses the **Louvain algorithm** (same modularity-optimization method Gephi uses under the hood) to assign each user to a community/cluster.
This gives you a `community_id` per user — cross-check this against Gephi's "Modularity Class" column to confirm they match.


In [ ]:
partition = community_louvain.best_partition(G)
community_col = pd.Series(partition, name="community_id")

print("Number of communities found:", community_col.nunique())
community_col.head(10)


## Feature 5 — Reciprocity

Fraction of a user's outgoing friend links that are mutual (i.e. A→B exists AND B→A exists).
Important to compute this from the **raw directed edge list**, before it gets treated as an undirected graph — otherwise you lose the asymmetry information entirely.


In [ ]:
edge_set = set(zip(user_friends["userID"], user_friends["friendID"]))

def fraction_mutual(u, out_edges_by_user):
    out_edges = out_edges_by_user.get(u, [])
    if not out_edges:
        return 0.0
    mutual = sum(1 for (a, b) in out_edges if (b, a) in edge_set)
    return mutual / len(out_edges)

# group outgoing edges per user once (faster than filtering edge_set repeatedly)
out_edges_by_user = {}
for u, f in zip(user_friends["userID"], user_friends["friendID"]):
    out_edges_by_user.setdefault(u, []).append((u, f))

reciprocity = pd.Series(
    {u: fraction_mutual(u, out_edges_by_user) for u in user_friends["userID"].unique()},
    name="reciprocity"
)

reciprocity.sort_values(ascending=False).head(10)


## Feature 6 — Reputation / Trustworthiness

Proxy: total plays a user has generated across all artists (`weight` column in `user_artists.dat`).
This reflects "popularity among listeners" — a heavier, more committed listener carries more behavioral trust signal.


In [ ]:
reputation = user_artists.groupby("userID")["weight"].sum().rename("total_plays")
reputation.sort_values(ascending=False).head(10)


## Step 3 — Merge everything into one feature table

One row per user, all 6 behavior features (as multiple sub-columns where relevant).


In [ ]:
final_features = pd.concat(
    [activity, popularity, pagerank, betweenness, community_col, reciprocity, reputation],
    axis=1
)
final_features.index.name = "userID"

print("Final shape:", final_features.shape)
final_features.head(10)


## Step 4 — Save to CSV

Export this so it's ready to merge with the other datasets' behavior feature tables later.


In [ ]:
final_features.to_csv("lastfm_behavior_features.csv")
print("Saved: lastfm_behavior_features.csv")


## Sanity checks (recommended before using this in your report)

- Compare `popularity_degree` totals against Gephi's Degree column for the same graph — they should match exactly.
- Compare `community_id` count/distribution against Gephi's Modularity Class output — Louvain has some randomness, so exact IDs may differ, but the *number* of communities and *grouping* of users should be very close.
- Check for users with `NaN` in `activity` or `reputation` columns — these are users present in the friend network but with no listening history (or vice versa), and you'll need to decide how to handle them (drop, or fill with 0).


In [ ]:
# Quick check: users with missing values after merge
final_features[final_features.isna().any(axis=1)]
